In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from sklearn.metrics import f1_score
from collections import defaultdict

### Datasets

In [ ]:
# Loadiong Data
value_set = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all.csv", sep='|')

value_train_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_train.csv", delimiter=',', dtype=int)
value_val_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_val.csv", delimiter=',', dtype=int)
value_test_set = np.loadtxt(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_test.csv", delimiter=',', dtype=int)

train_df = value_set.iloc[value_train_set]
val_df = value_set.iloc[value_val_set]
test_df = value_set.iloc[value_test_set]
test_df

### threshold by model

In [ ]:
# variables de control 
seeds = [0, 1, 2, 3, 4]
model_name = ["roberta-base","microsoft_deberta-v3-base"]
#threshold_grid = np.linspace(0.0, 1.0, 101)
#threshold_grid = np.linspace(0.2, 0.8, 501)  # step = 0.001
threshold_grid = np.linspace(0.05, 0.95, 181)  # step = 0.001

In [ ]:
# F1s per (model x value × threshold × seed)
f1_table = defaultdict(  # model
    lambda: defaultdict(  # seed
        lambda: defaultdict( # value
            lambda: defaultdict(list)  # threshold → list of F1s
        )
    )
)

for model in model_name:
    print("MODEL:",model)
    for seed in seeds:
        print("SEED:",seed)
        with open(f"/Proyecto/Value-disagreement/Python/Results/{model}_valueALL_seed{seed}.json") as f:
            data = json.load(f)
        #y_true = np.array(data["true"])
        #y_prob = np.array(data["preds"])

        test = val_df.copy()
        #test['preds'] = data['preds']#[x[0] for x in data['preds']]
        test['preds'] = [x[0] if isinstance(x, (list, tuple)) else x for x in data['preds']]
        test['true'] = data['true']
        test["prob"] = test["preds"].apply(lambda x: 1 / (1 + np.exp(-float(x))))
        
        """for val in test["value"].unique():
            for t in threshold_grid:
                y_bin = (test["prob"] >= t).astype(int)
                f1 = f1_score(test["label"], y_bin, zero_division=0)
                f1_table[model][seed][val][t].append(f1)"""
                
        for val in test["value"].unique():
            test_val = test[test["value"] == val]  # Filter for current value
            for t in threshold_grid:
                y_bin = (test_val["prob"] >= t).astype(int)
                f1 = f1_score(test_val["label"], y_bin, zero_division=0)
                f1_table[model][seed][val][t].append(f1)
f1_table

In [ ]:
test["prob"].describe()

In [ ]:
test.groupby("value")["prob"].describe()

In [ ]:
probs = pd.read_csv(r"/Proyecto/Value-disagreement/Python/Models/ensemble.probs.csv",
                     sep="|"
                     )
probs

In [ ]:
for val in probs["value"].unique():
    probs_val = probs[probs["value"] == val]  # Filter for current value
    for t in threshold_grid:
        y_bin = (probs_val["prob"] >= t).astype(int)
        f1 = f1_score(probs_val["label"], y_bin, zero_division=0)
        f1_table['ensemble']['bests'][val][t].append(f1)
f1_table

In [ ]:
# List of F1s across seeds
f1_avg_table = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

for model in f1_table:
    for seed in f1_table[model]:
        for val in f1_table[model][seed]:
            for t in f1_table[model][seed][val]:
                f1 = f1_table[model][seed][val][t][0]
                f1_avg_table[model][val][t].append(f1)

In [ ]:
best_thresholds = {}

for model in f1_avg_table:
    best_thresholds[model] = {}
    for val in f1_avg_table[model]:
        t_to_f1 = f1_avg_table[model][val]
        # Average across seeds
        avg_f1_per_t = {t: np.mean(f1s) for t, f1s in t_to_f1.items()}
        # Find the best threshold
        best_t = max(avg_f1_per_t, key=avg_f1_per_t.get)
        best_thresholds[model][val] = round(best_t, 3)
best_thresholds

In [ ]:
for val in test["value"].unique():
    f1s = [np.mean(f1_table[model][seed][val][t]) for t in threshold_grid]
    plt.plot(threshold_grid, f1s, label=val)
plt.legend()
plt.title(f"F1 vs Thresholds for {model}")
plt.show()

In [ ]:
# Save
with open("best_thresholds_per_model.json", "w") as f:
    json.dump(best_thresholds, f, indent=2)

### Ensembled

In [ ]:
# Inputs: assume these exist
# test_df: DataFrame with "value", "label"
# ensemble_probs: list of float probabilities for each row (same order as test_df)

# Thresholds to test
#threshold_grid = np.arange(0.05, 0.95, 0.01)
threshold_grid = np.linspace(0.3, 0.8, 501)  # step = 0.001

# Store best threshold per value
best_thresholds = {}

# Loop over value classes
for value in probs["value"].unique():
    subset = probs[probs["value"] == value].copy()
    prob = subset["prob"]  # use same row indices
    labels = subset["label"].values

    best_f1 = 0
    best_thresh = 0.5

    for t in threshold_grid:
        preds = (prob >= t).astype(int)
        f1 = f1_score(labels, preds, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t

    best_thresholds[value] = best_thresh
    print(f"Value: {value}, Best Threshold: {best_thresh:.2f}, F1: {best_f1:.3f}")

## Best model per class

In [ ]:
model_names = ["roberta-base", "microsoft_deberta-v3-base"]
seeds = [0]#, 1, 2, 3, 4]

In [ ]:
# Per-value F1s for each model
model_f1s = {model: defaultdict(list) for model in model_names}

# Load each JSON file and collect per-value F1s
for model in model_names:
    for seed in seeds:

        with open(f"/Proyecto/Value-disagreement/Python/Results/Per Value/{model}_valueALL_VAL_seed{seed}_per_value_f1.json") as f:
            f1_dict = json.load(f)

        for value, f1 in f1_dict.items():
            model_f1s[model][value].append(f1)
model_f1s

In [ ]:
# Average per-value F1 per model
avg_f1 = {
    model: {
        value: sum(f1_list) / len(f1_list)
        for value, f1_list in value_dict.items()
    }
    for model, value_dict in model_f1s.items()
}
avg_f1

In [ ]:
# Best model per value
best_model_per_value = {}
for value in avg_f1[model_names[0]].keys():
    f1_roberta = avg_f1["roberta-base"][value]
    f1_deberta = avg_f1["microsoft_deberta-v3-base"][value]
    best_model_per_value[value] = "roberta-base" if f1_roberta >= f1_deberta else "microsoft_deberta-v3-base"
best_model_per_value

In [ ]:
# Save
with open("best_model_per_value.json", "w") as f:
    json.dump(best_model_per_value, f, indent=2)